In [ ]:
%load_ext autoreload
%autoreload 2

import pandas as pd
import numpy as np
from scipy import stats
import statsmodels.api as sm
import statsmodels.formula.api as smf
import matplotlib.pyplot as plt

from img_labormarket.config import BLD, SRC
from img_labormarket.analysis.help_analysis import *
from img_labormarket.data_management.help_data import *
# Setze die Anzeigeoptionen für Pandas
pd.set_option('display.float_format', '{:.3f}'.format)

In [ ]:
df = pd.read_csv(BLD / 'data' / 'clean_data.csv')

# Demographics

In [ ]:

obs = df['img'].value_counts()

rslt = pd.DataFrame({'Native': f'{int(obs[0])}', 'Immigrant': f'{int(obs[1])}', 'p-value': [np.nan]},index=['Observations'])
rslt


In [ ]:
def compare_gender(df):
    # Split data
    df_img = df.loc[df['img'] == 1]
    df_nat = df.loc[df['img'] == 0]
    # COunt obs 
    df_img = df_img.loc[df_img['Gender']!= 'Prefer not to say']
    df_nat = df_nat.loc[df_nat['Gender']!= 'Prefer not to say']
    # Count obs
    count = np.array([
        df_nat['Gender'].value_counts().loc['Female'],
        df_img['Gender'].value_counts().loc['Female'],
                       ])
    # Observations 
    nobs = np.array([
        len(df_nat),
        len(df_img)])

    fraction = (count / nobs)*100
    z_stat, p_value = sm.stats.proportions_ztest(count, nobs, alternative='two-sided')

    p_value = '<0.001' if p_value < 0.001 else f'{p_value:.3f}'
    rslt = pd.DataFrame({'Native': f'{fraction[0]:.2f}', 'Immigrant': f'{fraction[1]:.2f}', 'p-value': p_value},index=['Female'])

    return rslt

rslt1 = compare_gender(df)
rslt1


In [ ]:
res = df.loc[df['img']==1]['born_region'].value_counts(normalize=True)*100
rslt2 = pd.DataFrame({'Native':[np.nan,np.nan],'Immigrant':res.values,'p-value':[np.nan,np.nan]},index=['Non-EU','EU'])
rslt2

In [ ]:
top = 5
res = (df.loc[df['img']==1]['born_country'].value_counts(normalize=True)*100).head(top)
rslt3 = pd.DataFrame({'Native':[np.nan]*top,'Immigrant':res.values,'p-value':[np.nan]*top},index=res.index)
rslt3

In [ ]:
def compare_age(df):
    # Split data
    #df['new_age'] = df['Age'].replace({'35-49 years':'35 years or older','50 years or older':'35 years or older'})
    df_img = df.loc[df['img'] == 1]
    df_nat = df.loc[df['img'] == 0]
    
    # Observations 
    nobs = np.array([
        len(df_nat),
        len(df_img)])

    rslt = pd.DataFrame({'Native':[],'Immigrant':[],'p-value':[]})
    for age in ['20-24 years','25-29 years','30-34 years','35 years or older']:
        # Count obs
        count = np.array([
            df_nat['new_age'].value_counts().loc[age],
            df_img['new_age'].value_counts().loc[age],
                       ])
        fraction = (count / nobs)*100
        z_stat, p_value = sm.stats.proportions_ztest(count, nobs, alternative='two-sided')

        p_value = '$<$0.001' if p_value < 0.001 else f'{p_value:.3f}'
        new_row = pd.DataFrame({'Native': f'{fraction[0]:.2f}', 'Immigrant': f'{fraction[1]:.2f}', 'p-value': p_value},index=[age])
        rslt = pd.concat([rslt,new_row])

    return rslt

rslt4 = compare_age(df)
rslt4


In [ ]:
def compare_study_field(df):
    # Split data
    df_img = df.loc[df['img'] == 1]
    df_nat = df.loc[df['img'] == 0]
    
    # Observations 
    nobs = np.array([
        len(df_nat),
        len(df_img)])

    rslt = pd.DataFrame({'Native':[],'Immigrant':[],'p-value':[]})
    for study_field in ['Business & Economics','Engineering & CS','Natural & Life Sciences']:
        # Count obs
        new_name = study_field.replace('&','\&')
        count = np.array([
            df_nat['StudyFieldCluster'].value_counts().loc[study_field],
            df_img['StudyFieldCluster'].value_counts().loc[study_field],
                       ])
        fraction = (count / nobs)*100
        z_stat, p_value = sm.stats.proportions_ztest(count, nobs, alternative='two-sided')

        p_value = '$<$0.001' if p_value < 0.001 else f'{p_value:.3f}'
        new_row = pd.DataFrame({'Native': f'{fraction[0]:.2f}', 'Immigrant': f'{fraction[1]:.2f}', 'p-value': p_value},index=[new_name])
        rslt = pd.concat([rslt,new_row])

    return rslt

rslt5 = compare_study_field(df)
rslt5

In [ ]:
def compare_degree(df):
    # Split data
    df_img = df.loc[df['img'] == 1]
    df_nat = df.loc[df['img'] == 0]
    
    # Observations 
    nobs = np.array([
        len(df_nat),
        len(df_img)])

    rslt = pd.DataFrame({'Native':[],'Immigrant':[],'p-value':[]})
    for degree in ['Bachelor','Master','PhD']:
        # Count obs
        count = np.array([
            df_nat['DegreeCompact'].value_counts().loc[degree],
            df_img['DegreeCompact'].value_counts().loc[degree],
                       ])
        fraction = (count / nobs)*100
        z_stat, p_value = sm.stats.proportions_ztest(count, nobs, alternative='two-sided')

        p_value = '$<$0.001' if p_value < 0.001 else f'{p_value:.3f}'
        new_row = pd.DataFrame({'Native': f'{fraction[0]:.2f}', 'Immigrant': f'{fraction[1]:.2f}', 'p-value': p_value},index=[degree])
        rslt = pd.concat([rslt,new_row])

    return rslt

rslt6 = compare_degree(df)
rslt6

In [ ]:
def compare_grad_year(df):
    df['new_grad'] = df['GradYear'].replace({2018:2020,2019:2020})
    # Split data
    df_img = df.loc[df['img'] == 1]
    df_nat = df.loc[df['img'] == 0]
    
    # Observations 
    nobs = np.array([
        len(df_nat),
        len(df_img)])

    rslt = pd.DataFrame({'Native':[],'Immigrant':[],'p-value':[]})
    for grad_year in range(2020,2025):
        # Count obs
        count = np.array([
            df_nat['new_grad'].value_counts().loc[grad_year],
            df_img['new_grad'].value_counts().loc[grad_year],
                       ])
        fraction = (count / nobs)*100
        z_stat, p_value = sm.stats.proportions_ztest(count, nobs, alternative='two-sided')

        p_value = '$<$0.001' if p_value < 0.001 else f'{p_value:.3f}'
        grad_year = f'{grad_year} or earlier' if grad_year == 2020 else f'{grad_year}'
        new_row = pd.DataFrame({'Native': f'{fraction[0]:.2f}', 'Immigrant': f'{fraction[1]:.2f}', 'p-value': p_value},index=[grad_year])
        rslt = pd.concat([rslt,new_row])

    return rslt

rslt7 = compare_grad_year(df)
rslt7

In [ ]:
rslt8 = native_img_comparison(df, ['GPA'])

#rslt8 = rslt8.rename(index=renamed_dict)
rslt8 = rslt8.rename(columns={'Immigrant Avg':'Immigrant','Native Avg':'Native','p-value':'p-value'})
rslt8[['Native','Immigrant', 'p-value']]

In [ ]:
rslt9 = native_img_comparison(df, ['Q60','Q61'])

renamed_dict = {
'Q60': 'Willingness to take risks (1-10)',
'Q61': 'Perceived relative ability (1-4)',
}
rslt9 = rslt9.rename(index=renamed_dict)
rslt9 = rslt9.rename(columns={'Immigrant Avg':'Immigrant','Native Avg':'Native','p-value':'p-value'})
rslt9['p-value'] = rslt9['p-value'].apply(lambda x: '$<$0.001' if x < 0.001 else f'{x:.3f}')
rslt9[['Native','Immigrant', 'p-value']]

In [ ]:
def generate_latex_table(rslt,rslt1,rslt2,rslt3,rslt4,rslt5,rslt6,rslt7,rslt8, filename="balance_table.tex"):
    """
    Generate a LaTeX table from a DataFrame with numerical values rounded
    to two decimal places and p-values to three decimal places.
    
    Parameters:
    df (pd.DataFrame): The data to format.
    filename (str): The output .tex file name.
    """
    # LaTeX Table Header
    latex_code = r"""
        \begin{tabular}{lccc}
            \toprule
             & Natives &  Immigrants & p-value \\
            \midrule
    """
    # Add rows dynamically
    for locs, row in rslt.iterrows():
        latex_code += f"      \\textbf{{{locs}}} & {row['Native']} & {row['Immigrant']} &  \\\ \n"
    latex_code += f"  \\textbf{{Gender}} &  & &  \\\ \n"
    for locs, row in rslt1.iterrows():
        latex_code += f"\\phantom{{00}}        {locs} & {row['Native']}\% & {row['Immigrant']}\% & {row['p-value']} \\\ \n"
    latex_code += f"  \\textbf{{Region}} &  & &  \\\ \n"
    for locs, row in rslt2.iterrows():
        latex_code += f"\\phantom{{00}}        {locs} &  & {row['Immigrant']}\% &  \\\ \n"
    latex_code += f"  \\textbf{{Countries of Origin (Top 5)}} &  & &  \\\ \n"
    for locs, row in rslt3.iterrows():
        latex_code += f"\\phantom{{00}}        {locs} &  & {row['Immigrant']:.2f}\% &  \\\ \n"
    latex_code += f"  \\textbf{{Age}} &  & &  \\\ \n"
    for locs, row in rslt4.iterrows():
        latex_code += f"\\phantom{{00}}        {locs} & {row['Native']}\% & {row['Immigrant']}\% & {row['p-value']} \\\ \n"
    latex_code += f"  \\textbf{{Degree}} &  & &  \\\ \n"
    for locs, row in rslt6.iterrows():
        latex_code += f"\\phantom{{00}}        {locs} & {row['Native']}\% & {row['Immigrant']}\% & {row['p-value']} \\\ \n"
    latex_code += f"  \\textbf{{Field of Study}} &  & &  \\\ \n"
    for locs, row in rslt5.iterrows():
        latex_code += f"\\phantom{{00}}        {locs} & {row['Native']}\% & {row['Immigrant']}\% & {row['p-value']} \\\ \n"
    latex_code += f"  \\textbf{{Graduation Year}} &  & &  \\\ \n"
    for locs, row in rslt7.iterrows():
        latex_code += f"\\phantom{{00}}        {locs} & {row['Native']}\% & {row['Immigrant']}\% & {row['p-value']} \\\ \n"
    for locs, row in rslt8.iterrows():
        latex_code += f"        \\textbf{{{locs}}} & {float(row['Native']):.2f} & {float(row['Immigrant']):.2f} & {float(row['p-value']):.3f} \\\ \n"
    latex_code += f"  \\textbf{{Personality}} &  & &  \\\ \n"
    for locs, row in rslt9.iterrows():
        latex_code += f"\\phantom{{00}}        {locs} & {float(row['Native']):.2f} & {float(row['Immigrant']):.2f} & {row['p-value']} \\\ \n"
    # LaTeX Table Footer
    latex_code += r"""        \bottomrule
    \end{tabular}
    """
    
    # Save to .tex file
    with open(filename, "w") as f:
        f.write(latex_code)
    
    return filename

generate_latex_table(rslt,rslt1,rslt2,rslt3,rslt4,rslt5,rslt6,rslt7,rslt8, filename=BLD / "tables" / "descriptives.tex")

In [ ]:
def generate_latex_table_p1(rslt,rslt1,rslt2,rslt3,rslt4,rslt5,rslt6,rslt7,rslt8, filename="balance_table.tex"):
    """
    Generate a LaTeX table from a DataFrame with numerical values rounded
    to two decimal places and p-values to three decimal places.
    
    Parameters:
    df (pd.DataFrame): The data to format.
    filename (str): The output .tex file name.
    """
    # LaTeX Table Header
    latex_code = r"""
        \begin{tabular}{lccc}
            \toprule
             & Natives &  Immigrants & p-value \\
            \midrule
    """
    # Add rows dynamically
    for locs, row in rslt.iterrows():
        latex_code += f"        \\textbf{{{locs}}} & {row['Native']} & {row['Immigrant']} &  \\\ \n"
    latex_code += f"  \\textbf{{Gender}} &  & &  \\\ \n"
    for locs, row in rslt1.iterrows():
        latex_code += f"\\phantom{{00}}         {locs} & {row['Native']}\% & {row['Immigrant']}\% & {row['p-value']} \\\ \n"
    latex_code += f"  \\textbf{{Region}} &  & &  \\\ \n"
    for locs, row in rslt2.iterrows():
        latex_code += f"\\phantom{{00}}         {locs} &  & {row['Immigrant']}\% &  \\\ \n"
    latex_code += f"  \\textbf{{Countries of Origin (Top 5)}} &  & &  \\\ \n"
    for locs, row in rslt3.iterrows():
        latex_code += f"\\phantom{{00}}         {locs} &  & {row['Immigrant']:.2f}\% &  \\\ \n"
    latex_code += f"  \\textbf{{Age}} &  & &  \\\ \n"
    for locs, row in rslt4.iterrows():
        latex_code += f"\\phantom{{00}}         {locs} & {row['Native']}\% & {row['Immigrant']}\% & {row['p-value']} \\\ \n"
    # LaTeX Table Footer
    latex_code += r"""        \bottomrule
    \end{tabular}
    """
    
    # Save to .tex file
    with open(filename, "w") as f:
        f.write(latex_code)
    
    return filename

def generate_latex_table_p2(rslt,rslt1,rslt2,rslt3,rslt4,rslt5,rslt6,rslt7,rslt8, filename="balance_table.tex"):
    """
    Generate a LaTeX table from a DataFrame with numerical values rounded
    to two decimal places and p-values to three decimal places.
    
    Parameters:
    df (pd.DataFrame): The data to format.
    filename (str): The output .tex file name.
    """
    # LaTeX Table Header
    latex_code = r"""
        \begin{tabular}{lccc}
            \toprule
             & Natives &  Immigrants & p-value \\
            \midrule
    """
    # Add rows dynamically
    for locs, row in rslt.iterrows():
        latex_code += f"        \\textbf{{{locs}}} & {row['Native']} & {row['Immigrant']} &  \\\ \n"
    latex_code += f"  \\textbf{{Degree}} &  & &  \\\ \n"
    for locs, row in rslt6.iterrows():
        latex_code += f"\\phantom{{00}}         {locs} & {row['Native']}\% & {row['Immigrant']}\% & {row['p-value']} \\\ \n"
    latex_code += f"  \\textbf{{Field of Study}} &  & &  \\\ \n"
    for locs, row in rslt5.iterrows():
        latex_code += f"\\phantom{{00}}         {locs} & {row['Native']}\% & {row['Immigrant']}\% & {row['p-value']} \\\ \n"
    latex_code += f"  \\textbf{{Graduation Year}} &  & &  \\\ \n"
    for locs, row in rslt7.iterrows():
        latex_code += f"\\phantom{{00}}         {locs} & {row['Native']}\% & {row['Immigrant']}\% & {row['p-value']} \\\ \n"
    for locs, row in rslt8.iterrows():
        latex_code += f"        \\textbf{{{locs}}} & {float(row['Native']):.2f} & {float(row['Immigrant']):.2f} & {float(row['p-value']):.3f} \\\ \n"
    latex_code += f"  \\textbf{{Personality}} &  & &  \\\ \n"
    for locs, row in rslt9.iterrows():
        latex_code += f"\\phantom{{00}}         {locs} & {float(row['Native']):.2f} & {float(row['Immigrant']):.2f} & {row['p-value']} \\\ \n"
    # LaTeX Table Footer
    latex_code += r"""        \bottomrule
    \end{tabular}
    """
    
    # Save to .tex file
    with open(filename, "w") as f:
        f.write(latex_code)
    
    return filename

generate_latex_table_p1(rslt,rslt1,rslt2,rslt3,rslt4,rslt5,rslt6,rslt7,rslt8, filename=BLD / "tables" / "descriptives_p1.tex")
generate_latex_table_p2(rslt,rslt1,rslt2,rslt3,rslt4,rslt5,rslt6,rslt7,rslt8, filename=BLD / "tables" / "descriptives_p2.tex")